In [33]:
import pandas as pd
import numpy as np

---

### 🚨 ¿Cuándo crear un nuevo DataFrame? (View vs. Copy)

Cuando filtras en Pandas, a veces obtienes una "Vista" (como mirar a través de una ventana al DataFrame original) y a veces una "Copia" (un DataFrame completamente nuevo e independiente). Confundir esto genera el temido error **`SettingWithCopyWarning`**.

* **NO crear un DataFrame nuevo (Usar una Vista):** Cuando solo quieres *mirar* los datos, calcular un promedio rápido, o exportarlos inmediatamente.
* **SÍ crear un DataFrame nuevo (Usar `.copy()`):** Cuando vas a filtrar los datos y luego vas a **modificar o agregar columnas** a esos datos filtrados.

**Ejemplo Práctico:**

In [34]:
# Simulamos datos sucios extraídos de una tienda de hardware
df = pd.DataFrame({
    'tienda' : ['CompraGamer', 'Mercado Libre', 'CompraGamer'],
    'producto': ['RTX 4060', 'Ryzen 5', 'Gabinete ATX'],
    'categoria' : ['Placas de Video', 'Procesador', 'Gabinete'],
    'estado' : ['nuevo', 'nuevo', 'usado'],
    'precio': [450000.0, 340000.0, 85000.0],
    'stock': [15, 10, 10]
})

# ❌ MAL: Pandas no sabe si 'df_gpus' es una vista o una copia. 
# Si luego haces df_gpus['nueva_col'] = X, te lanzará el SettingWithCopyWarning.
df_gpus = df[df['categoria'] == 'Placas de Video'] 

# ✅ BIEN: Solo para mirar o leer (No asignamos a variable, o no lo modificamos después)
print(df[df['precio'] > 100000]['precio'].mean())

# ✅ PRO: Crear explícitamente un nuevo DataFrame para trabajarlo de forma segura
df_gpus_limpio = df[df['categoria'] == 'Placas de Video'].copy()
df_gpus_limpio['precio_con_iva'] = df_gpus_limpio['precio'] * 1.21 # Esto ahora es 100% seguro
df_gpus_limpio

395000.0


,tienda,producto,categoria,estado,precio,stock,precio_con_iva
0,CompraGamer,RTX 4060,Placas de Video,nuevo,450000.0,15,544500.0


---

### 1. 🎯 `.loc[]`: El Estándar Profesional

Es el método más robusto y explícito. Te permite filtrar filas y seleccionar columnas al mismo tiempo.

| Parámetro / Sintaxis | ¿Para qué sirve? |
| --- | --- |
| `[condición_filas, lista_columnas]` | El primer argumento filtra qué filas quieres, el segundo qué columnas te traes. |
| `&` (AND), `|` (OR), `~` (NOT) | Operadores lógicos. **Obligatorio** usar paréntesis `()` en cada condición. |

**Ejemplo Práctico:**
Imagina que quieres buscar procesadores Ryzen con stock en una tienda específica, y solo necesitas ver el componente y el precio.

In [35]:
# Filtramos filas con múltiples condiciones y seleccionamos solo 2 columnas
df_ryzen = df.loc[
    (df['producto'].str.contains('Ryzen')) & (df['stock'] > 0), 
    ['producto', 'precio']
].copy() # Usamos .copy() porque seguramente luego querremos limpiar esos precios

df_ryzen

,producto,precio
1,Ryzen 5,340000.0


---

### 2. ⚡ `.query()`: El Método Elegante (Ideal para Pipelines)

Si tienes muchas condiciones, `.loc` se vuelve difícil de leer por tantos corchetes y paréntesis. `.query()` permite escribir el filtro como un string (texto), similar a la cláusula `WHERE` en SQL. Es ligeramente más rápido en datasets de más de 200,000 filas.

| Parámetro Clave | ¿Para qué sirve? |
| --- | --- |
| `expr` (String) | La expresión lógica. Usas `and`, `or`, `not` en lugar de símbolos. |
| `@variable` | El prefijo `@` te permite inyectar variables de Python directamente en el string del query. |

**Ejemplo Práctico:**



In [36]:
presupuesto_max = 350000
categoria_buscada = 'Procesador'

# Código súper limpio y legible. Usamos @ para inyectar las variables
df_ofertas = df.query(
    "precio <= @presupuesto_max and categoria == @categoria_buscada and stock > 0"
).copy()

df_ofertas

,tienda,producto,categoria,estado,precio,stock
1,Mercado Libre,Ryzen 5,Procesador,nuevo,340000.0,10


---

### 3. 🔍 `.isin()`: El Filtro de Listas Clave

Perfecto cuando necesitas filtrar filas que coincidan con múltiples valores exactos, evitando hacer una cadena enorme de condiciones `OR`.

| Uso | ¿Qué hace? |
| --- | --- |
| `df['columna'].isin([lista])` | Devuelve `True` si el valor de la celda está dentro de la lista proporcionada. |
| `~df['columna'].isin([lista])` | La tilde `~` invierte la lógica (excluye los elementos de la lista). |

**Ejemplo Práctico:**


In [41]:
# Queremos analizar precios solo de tiendas de confianza
tiendas_confianza = ['CompraGamer', 'Venex', 'FullH4rd']

df_confiable = df.loc[df['tienda'].isin(tiendas_confianza)].copy()
print(f"Tiendas confiables:\n{df_confiable}")
# O al revés: queremos excluir componentes usados
categorias_excluir = ['reacondicionado', 'outlet', 'usado']
df_nuevos = df.loc[~df['estado'].isin(categorias_excluir)].copy()
print(f"\nProductos Nuevos:\n{df_nuevos}")

Tiendas confiables:
        tienda      producto        categoria estado    precio  stock
0  CompraGamer      RTX 4060  Placas de Video  nuevo  450000.0     15
2  CompraGamer  Gabinete ATX         Gabinete  usado   85000.0     10

Productos Nuevos:
          tienda  producto        categoria estado    precio  stock
0    CompraGamer  RTX 4060  Placas de Video  nuevo  450000.0     15
1  Mercado Libre   Ryzen 5       Procesador  nuevo  340000.0     10


---

### 4. 📝 `.str.contains()`: El Salvador del Web Scraping

Cuando raspas datos de e-commerce, los títulos de los productos son un caos de texto. Este método aplica una búsqueda de texto (incluso con Expresiones Regulares) a toda la columna.

| Parámetro | Opciones / Uso | ¿Qué hace? |
| --- | --- | --- |
| `pat` | String o Regex | La palabra o patrón que estás buscando. |
| `case` | `False` / `True` (default) | Si es `False`, ignora mayúsculas y minúsculas. ¡Vital para textos sucios! |
| `na` | `False` | Si hay valores nulos (`NaN`), devolverá `False` en lugar de romper el filtro. |

**Ejemplo Práctico:**

In [43]:
# Busca cualquier GPU que sea 4060 o 4070, ignorando si dice "rtx", "Rtx" o "RTX"
df_gpus_midrange = df.loc[
    df['producto'].str.contains('4060|4070', case=False, na=False)
].copy()

df_gpus_midrange

,tienda,producto,categoria,estado,precio,stock
0,CompraGamer,RTX 4060,Placas de Video,nuevo,450000.0,15


---

### 💡 Resumen de Buenas Prácticas para tu Pipeline

1. **Usa `.loc`** para asignaciones y cruces de columnas.
2. **Usa `.query()`** cuando el código empiece a verse feo y difícil de leer.
3. **Usa `.copy()`** al final de un filtro fuerte si ese nuevo sub-dataset es el que vas a usar para el resto del script.